In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from tqdm import trange
import copy
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [2]:
from main_alg import main_alg

In [8]:
def make_singular(n, p, rng=0, noise=0.001,radius=1):

    #matrix A
    singular_part=int(p/5)
    rng = np.random.default_rng(rng)
    A = rng.normal(size=(p+1, n))
    A[singular_part:2*singular_part,:]=50*rng.normal(size=(singular_part,n))
    preprocess = StandardScaler()
    A = preprocess.fit_transform(A)
    A=radius*np.transpose(A)/np.sqrt(p)

    #hidden x
    x = np.zeros(p)
    x[singular_part:2*singular_part] = 3
    A[:,p] = A[:,0:-1] @ x + noise*rng.normal(size=n)



    return A, x

Run experiments

In [3]:
c1=0.01
c2=0.1

In [4]:
dimension=64
n=10000

In [5]:
candidate_epsilons=[0.1,0.5,1]
candidate_deltas=[0.1,0.3,0.5]
data_noises=[0.01,0.1,1]

In [6]:
exp_data=[]

In [ ]:
i=0
for data_noise in (data_noises):
  datastream,x=make_singular(n,dimension,noise=data_noise)
  random_order=np.random.permutation(datastream)
  for epsilon in (candidate_epsilons):
    for delta in (candidate_deltas):
      U=2*np.log2(1/delta)/epsilon+1
      model=main_alg(epsilon,delta,U,dimension,random_order,c1,c2,eta=0.05/U)
      model.simulate(iter=1)
      exp_data.append(copy.deepcopy([model.scores_users,model.scores_batches]))
      i+=1
      print(f'finish {i} round')

In [ ]:
for i,data_noise in enumerate(data_noises):
  for j,epsilon in enumerate(candidate_epsilons):
    plt.figure()
    for k,delta in enumerate(candidate_deltas):
      plt.plot(exp_data[4*i+2*j+k][0],label=f'$\delta={delta}$')
    plt.rcParams.update({'font.size': 16})
    plt.title(f'Objective Value w.r.t. Users, $\sigma={data_noise}, \epsilon={epsilon}$')
    plt.xlabel('Number of Users')
    plt.ylabel('Objective Value')
    plt.xscale('log',base=2)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
    plt.savefig(f'figure-randbin/users-sigma={data_noise},epsilon={epsilon}.pdf', bbox_inches = 'tight')
    plt.show()

In [ ]:
for i,data_noise in enumerate(data_noises):
  for k,delta in enumerate(candidate_deltas):
    plt.figure()
    for j,epsilon in enumerate(candidate_epsilons):
      plt.plot(exp_data[4*i+2*j+k][0],label=f'$\epsilon={epsilon}$')
    plt.rcParams.update({'font.size': 16})
    plt.title(f'Objective Value w.r.t. Users, $\sigma={data_noise}, \delta={delta}$')
    plt.xlabel('Number of Users')
    plt.ylabel('Objective Value')
    plt.xscale('log',base=2)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
    plt.savefig(f'figure-randbin/users-sigma={data_noise},delta={delta}.pdf', bbox_inches = 'tight')
    plt.show()

In [ ]:
for i,data_noise in enumerate(data_noises):
  for j,epsilon in enumerate(candidate_epsilons):
    plt.figure()
    for k,delta in enumerate(candidate_deltas):
      plt.plot(exp_data[4*i+2*j+k][1],label=f'$\delta={delta}$')
    plt.rcParams.update({'font.size': 16})
    plt.title(f'Objective Value w.r.t. Batches, $\sigma={data_noise}, \epsilon={epsilon}$')
    plt.xlabel('Number of Batches')
    plt.ylabel('Objective Value')
    #plt.xscale('log',base=2)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
    plt.savefig(f'figure-randbin/batches-sigma={data_noise},epsilon={epsilon}.pdf', bbox_inches = 'tight')
    plt.show()

In [ ]:
for i,data_noise in enumerate(data_noises):
  for k,delta in enumerate(candidate_deltas):
    plt.figure()
    for j,epsilon in enumerate(candidate_epsilons):
      plt.plot(exp_data[4*i+2*j+k][1],label=f'$\epsilon={epsilon}$')
    plt.rcParams.update({'font.size': 16})
    plt.title(f'Objective Value w.r.t. Batches, $\sigma={data_noise}, \delta={delta}$')
    plt.xlabel('Number of Batches')
    plt.ylabel('Objective Value')
    #plt.xscale('log',base=2)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
    plt.savefig(f'figure-randbin/batches-sigma={data_noise},delta={delta}.pdf', bbox_inches = 'tight')
    plt.show()

Comparation between random batch size and fixed batch size

In [ ]:
data_noise=1
candidate_epsilons=[0.1,0.5,1]
candidate_deltas=[0.1,0.3,0.5]

In [ ]:
exp_data_fix_batch=[]
exp_data=[]

In [ ]:
i=0
datastream,x=make_singular(n,dimension,noise=data_noise)
random_order=np.random.permutation(datastream)
for epsilon in (candidate_epsilons):
  for delta in (candidate_deltas):
    U=2*np.log2(1/delta)/epsilon+1
    model=main_alg(epsilon,delta,U,dimension,random_order,c1,c2,eta=0.05/U)
    model.simulate(iter=1)
    exp_data.append(copy.deepcopy([model.scores_users,model.scores_batches]))
    model=main_alg(epsilon,delta,U,dimension,random_order,c1,c2,eta=0.05/U)
    model.simulate_fix_size()
    exp_data_fix_batch.append(copy.deepcopy([model.fix_batch_scores_users,model.fix_batch_scores_batches]))
    i+=1
    print(f'finish {i} round')

In [ ]:
for k,delta in enumerate(candidate_deltas):

  plt.figure()
  for j,epsilon in enumerate(candidate_epsilons):
    plt.plot(exp_data[3*j+k][0],label=f'$\epsilon={epsilon}$')
    plt.plot(exp_data_fix_batch[3*j+k][0],label=f'FS, $\epsilon={epsilon}$')
  plt.rcParams.update({'font.size': 16})
  plt.title(f'Objective Value w.r.t. Users, $\sigma={data_noise}, \delta={delta}$')
  plt.xlabel('Number of Users')
  plt.ylabel('Objective Value')
  #plt.xscale('log',base=2)
  plt.legend(loc='upper right')
  plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
  plt.savefig(f'figure-randbin/fix-batch-users-sigma={data_noise},delta={delta}.pdf', bbox_inches = 'tight')
  plt.show()

In [ ]:

for j,epsilon in enumerate(candidate_epsilons):
  plt.figure()
  for k,delta in enumerate(candidate_deltas):
    plt.plot(exp_data[3*j+k][0],label=f'$\delta={delta}$')
    plt.plot(exp_data_fix_batch[3*j+k][0],label=f'FS, $\delta={delta}$')
  plt.rcParams.update({'font.size': 16})
  plt.title(f'Objective Value w.r.t. Users, $\sigma={data_noise}, \epsilon={epsilon}$')
  plt.xlabel('Number of Users')
  plt.ylabel('Objective Value')
  #plt.xscale('log',base=2)
  plt.legend(loc='upper right')
  plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
  plt.savefig(f'figure-randbin/fix-batch-users-sigma={data_noise},epsilon={epsilon}.pdf', bbox_inches = 'tight')
  plt.show()

In [ ]:
for k,delta in enumerate(candidate_deltas):

  plt.figure()
  for j,epsilon in enumerate(candidate_epsilons):
    plt.plot(exp_data[3*j+k][1],label=f'$\delta={delta}$')
    plt.plot(exp_data_fix_batch[3*j+k][1],label=f'FS, $\delta={delta}$')
  plt.rcParams.update({'font.size': 16})
  plt.title(f'Objective Value w.r.t. Batches, $\sigma={data_noise}, \epsilon={epsilon}$')
  plt.xlabel('Number of Batches')
  plt.ylabel('Objective Value')
  #plt.xscale('log',base=2)
  plt.legend(loc='upper right')
  plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
  plt.savefig(f'figure-randbin/fix-batch-batches-sigma={data_noise},delta={delta}.pdf', bbox_inches = 'tight')
  plt.show()

In [ ]:

for j,epsilon in enumerate(candidate_epsilons):
  plt.figure()

  for k,delta in enumerate(candidate_deltas):
    plt.plot(exp_data[3*j+k][1],label=f'$\delta={delta}$')
    plt.plot(exp_data_fix_batch[3*j+k][1],label=f'FS, $\delta={delta}$')
  plt.rcParams.update({'font.size': 16})
  plt.title(f'Objective Value w.r.t. Batches, $\sigma={data_noise}, \epsilon={epsilon}$')
  plt.xlabel('Number of Batches')
  plt.ylabel('Objective Value')
  #plt.xscale('log',base=2)
  plt.legend(loc='upper right')
  plt.grid(True, linestyle='--', alpha=0.5,zorder=0)
  plt.savefig(f'/content/drive/MyDrive/figure-randbin/fix-batch-batches-sigma={data_noise},epsilon={epsilon}.pdf', bbox_inches = 'tight')
  plt.show()